## WP013 stage 3 — confirmation on new held-out matches

**Heavy compute — run this yourself (about 65–70 min for the two arms at `MAX_WORKERS=3`).** See `README.md`, "Stage 3", for the design and the pre-declared test.

The 35 fits are the same as stage 2 (same `train_end`, same seeds), but each window now scores rounds `train_end + 2` … `train_end + 5`: **1,469 matches never scored before**, none of which the continuity idea was selected on. Two arms are enough: `baseline` and `continuity`.

**Primary (declared before running):** paired RPS `continuity − baseline` on the new matches with Pinnacle closing odds; a hit needs the 95% CI entirely below zero **and** a negative mean in both halves of the data.

In [ ]:
import pickle, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

from football_model.evaluation import market as mk

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'
WP013 = REPO / 'work_products' / 'wp013_lineup_continuity'
SCRIPT = REPO / 'scripts' / 'run_cv_window.py'
sys.path.insert(0, str(REPO / 'scripts'))
from run_cv_window import run_windows_concurrent  # noqa: E402

DATA_PATH = WP013 / 'cv_shared_data_confirm.pkl'      # same fits as stage 2, longer test spans
with open(DATA_PATH, 'rb') as f:
    shared = pickle.load(f)
df_cv, windows = shared['df_cv'], shared['windows']
odds = pd.read_pickle(WP003 / 'odds_raw.pkl').dropna(subset=['Date', 'FTR']).reset_index(drop=True)

per_round = df_cv[df_cv['is_home'] == 1].groupby('round').size()
new_rounds = sorted({r for w in windows for r in range(w['test_start'], w['test_end'] + 1)})
print(len(windows), 'windows;', len(new_rounds), 'new test rounds;', int(per_round.reindex(new_rounds).sum()), 'new matches')
print('example spans:', windows[0], windows[24], windows[-1])

ARMS = {'baseline': {}, 'continuity': {'use_continuity': True}}
MAX_WORKERS = 3
WINDOW_TIMEOUT = 1800

def load_ckpt(p):
    p = Path(p)
    return pickle.load(open(p, 'rb')) if p.exists() else {'results': [], 'cv_match_predictions': []}

In [ ]:
# Time ONE window first (window 25 took ~2.8 min on a loaded machine; ~35 s idle).
t0 = time.time()
run_windows_concurrent(SCRIPT, DATA_PATH, WP013 / 'cv_checkpoint_confirm_continuity.pkl', [25],
                       config_overrides=ARMS['continuity'], max_workers=1, timeout=WINDOW_TIMEOUT)
print(f'one window: {time.time() - t0:.1f}s')

In [ ]:
FULL_WINDOWS = list(range(1, len(windows) + 1))
t0 = time.time()
for name, ov in ARMS.items():
    run_windows_concurrent(SCRIPT, DATA_PATH, WP013 / f'cv_checkpoint_confirm_{name}.pkl', FULL_WINDOWS,
                           config_overrides=ov, max_workers=MAX_WORKERS, timeout=WINDOW_TIMEOUT)
print(f'\nwall time: {(time.time() - t0) / 60:.1f} min')
for name in ARMS:
    c = load_ckpt(WP013 / f'cv_checkpoint_confirm_{name}.pkl')
    print(f'  {name}: {len(c["results"])}/{len(FULL_WINDOWS)} windows, {len(c["cv_match_predictions"])} matches')

### Analysis

Joins each arm's predictions to Pinnacle's closing odds (the join raises if a score disagrees with the odds file; `model_fixtures` raises if a prediction is misaligned to its fixture) and reports the primary test. Everything after the verdict is context, not part of the test.

In [ ]:
need = [f'PSC{o}' for o in mk.OUTCOMES]
ckpt = {n: load_ckpt(WP013 / f'cv_checkpoint_confirm_{n}.pkl') for n in ARMS}
assert all(len(c['results']) == len(windows) for c in ckpt.values()), 'run all 35 windows for both arms first'

J = {n: mk.join_odds(mk.model_fixtures(df_cv, windows, c), odds, required_cols=need) for n, c in ckpt.items()}
keys = J['baseline'][['date', 'home_fd', 'away_fd']]
assert J['continuity'][['date', 'home_fd', 'away_fd']].equals(keys), 'arms cover different matches'

y = mk.outcome_onehot(J['baseline']['FTR'])
pin = mk.devig(mk.odds_matrix(J['baseline'], 'PSC'))
R = {n: mk.rps(j[['p_home_model', 'p_draw_model', 'p_away_model']].to_numpy(), y) for n, j in J.items()}
rp = mk.rps(pin, y)
print(f'{len(y)} new matches with Pinnacle closing odds; Pinnacle RPS {rp.mean():.4f}')
for n in ARMS:
    m, lo, hi = mk.bootstrap_ci(R[n] - rp, 5000)
    print(f'  {n:<11} RPS {R[n].mean():.4f}  gap to Pinnacle {m:+.4f} [{lo:+.4f}, {hi:+.4f}]   (2-5 rounds ahead: not comparable with stage 2)')

d = R['continuity'] - R['baseline']
mean, lo, hi = mk.bootstrap_ci(d, 5000)
first, second = mk.half_masks(J['baseline']['date'])
h1, h2 = d[first].mean(), d[second].mean()
hit = (hi < 0) and (h1 < 0) and (h2 < 0)
print(f'\nPRIMARY continuity − baseline: {mean:+.5f}  95% CI [{lo:+.5f}, {hi:+.5f}]   halves {h1:+.5f} / {h2:+.5f}   ->  {"HIT" if hit else "no hit"}')
print(f'continuity better on {(d < 0).mean():.1%} of matches')

### Context (not part of the test)

In [ ]:
# Reproducibility: the fits are identical to stage 2's (same train_end, same seeds), so beta_continuity should match window by window.
prev = load_ckpt(WP013 / 'cv_checkpoint_full_continuity.pkl')
b_new = {r['window']: r['beta_continuity'] for r in ckpt['continuity']['results']}
b_old = {r['window']: r['beta_continuity'] for r in prev['results']}
diff = np.array([b_new[w] - b_old[w] for w in sorted(b_new) if w in b_old])
print(f'beta_continuity vs stage 2, {len(diff)} windows: max abs difference {np.abs(diff).max():.2e}   (mean now {np.mean(list(b_new.values())):+.4f})')

# Calendar-year breakdown of the primary difference.
yr = pd.to_datetime(J['baseline']['date']).dt.year
print(pd.DataFrame({'year': yr, 'diff': d}).groupby('year')['diff'].agg(['count', 'mean']).round(5).T.to_string())

# Pooled with stage 2's 401 matches: OPTIMISTIC, since those are the matches the idea was selected on.
sh2 = pickle.load(open(WP013 / 'cv_shared_data.pkl', 'rb'))
old = {n: mk.join_odds(mk.model_fixtures(sh2['df_cv'], sh2['windows'], load_ckpt(WP013 / f'cv_checkpoint_full_{n}.pkl')), odds, required_cols=need)
       for n in ARMS}
yo = mk.outcome_onehot(old['baseline']['FTR'])
d_old = (mk.rps(old['continuity'][['p_home_model', 'p_draw_model', 'p_away_model']].to_numpy(), yo)
         - mk.rps(old['baseline'][['p_home_model', 'p_draw_model', 'p_away_model']].to_numpy(), yo))
pooled = np.concatenate([d, d_old])
print(f'\nstage-2 matches alone: {d_old.mean():+.5f} (n={len(d_old)});  pooled: {mk.bootstrap_ci(pooled, 5000)[0]:+.5f} '
      f'[{mk.bootstrap_ci(pooled, 5000)[1]:+.5f}, {mk.bootstrap_ci(pooled, 5000)[2]:+.5f}] (n={len(pooled)})  -- exploratory only')